# 🎲 Making ChatGPT Talk: Sampling Methods & Evaluation

Welcome back! In the last notebook, we trained ChatGPT. Now comes the fun part: **how does it actually pick words when talking to you?**

Spoiler: it doesn't just pick the most likely word every time. That would be *really* boring. And kind of broken.

## What You'll Learn

By the end of this notebook, you'll understand:
- 🔁 **Why greedy/beam search fails** for chatbots (hello, repetition!)
- 🎰 **Multinomial sampling** — random, but with structure
- 🔝 **Top-k sampling** — trim the probability distribution
- 🌐 **Top-p (nucleus) sampling** — dynamic trimming that's smarter
- 🌡️ **Temperature** — the "spice dial" for creativity
- ⛔ **Repetition penalty** — stopping the robot from getting stuck
- 📊 **Benchmarks** — MMLU, HumanEval, TruthfulQA
- 🛡️ **Safety evaluation** — toxicity, bias, adversarial robustness
- 🏗️ **System design** — the full ChatGPT pipeline

**Time:** ~60 minutes

---

## 🚫 Why Deterministic Methods Fail for Chatbots

### ELI12: The Broken Record 🎵

Imagine you ask a friend to finish the sentence: *"I like to eat..."*

A **greedy** friend would always say the *single most popular* answer: "pizza."
Ask again: "pizza."
Ask AGAIN: "pizza."
Forever. "pizza."

A **beam search** friend tries multiple branches at once (like planning 3 moves ahead in chess), but still tends toward the "safe" answers... and weirdly, once they start a pattern, they repeat it until it breaks your brain.

### Staff-Level: The Degeneration Problem

**Greedy decoding:** At each step, pick `argmax P(next_token | context)`. Deterministic. Fast. But:
- Leads to **repetitive loops** — high-probability n-grams get reinforced
- No diversity — every run gives the exact same output
- Optimizes local decisions, not global coherence

**Beam search** (keep top-B hypotheses at each step):
- Better quality on machine translation (short, factual output)
- Catastrophically repetitive for open-ended generation
- The [Holtzman et al. 2020 "Curious Case of Neural Text Degeneration"](https://arxiv.org/abs/1904.09751) paper proved this empirically

```
Why beam search fails for chat (intuition):

  At each step, the model assigns highest probability to
  "safe", generic continuations. Beam search amplifies this
  by always keeping those safe beams.

  Result: The model gets stuck in a local optimum of
  high-probability but low-quality text.

  Example beam search output:
  "I think it's important to note that it's important to
   note that the most important thing is that it is
   important to note that..."
```

**Solution:** Introduce *controlled randomness* via sampling. 🎲

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn.functional as F

# ============================================================
# Demo: Greedy Search Repetition Problem
# ============================================================
# We'll simulate a tiny bigram language model to show
# how greedy decoding creates repetitive loops.

# Vocabulary
vocab = ["the", "cat", "sat", "on", "mat", "and", ".", "then", "<EOS>"]
vocab_size = len(vocab)
w2i = {w: i for i, w in enumerate(vocab)}
i2w = {i: w for i, w in enumerate(vocab)}

# Bigram transition probabilities (manually crafted to show the problem)
# Row = current token, Col = next token
# Designed so "the" -> "cat" -> "sat" -> "on" -> "the" is a loop!
transitions = np.array([
    #  the   cat   sat    on   mat   and    .   then  EOS
    [0.00, 0.60, 0.10, 0.05, 0.10, 0.05, 0.05, 0.04, 0.01],  # the
    [0.00, 0.00, 0.70, 0.10, 0.10, 0.05, 0.04, 0.01, 0.00],  # cat
    [0.00, 0.00, 0.00, 0.75, 0.10, 0.05, 0.05, 0.04, 0.01],  # sat
    [0.50, 0.10, 0.00, 0.00, 0.15, 0.10, 0.10, 0.04, 0.01],  # on
    [0.05, 0.05, 0.00, 0.05, 0.00, 0.40, 0.30, 0.10, 0.05],  # mat
    [0.35, 0.15, 0.05, 0.05, 0.10, 0.00, 0.10, 0.15, 0.05],  # and
    [0.10, 0.10, 0.00, 0.00, 0.05, 0.10, 0.05, 0.50, 0.10],  # .
    [0.20, 0.20, 0.10, 0.10, 0.10, 0.10, 0.10, 0.00, 0.10],  # then
    [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00],  # EOS
])


def greedy_generate(start_token="the", max_len=25):
    """Greedy decoding: always pick the most likely next token."""
    current = w2i[start_token]
    tokens = [current]
    for _ in range(max_len):
        if i2w[current] == "<EOS>":
            break
        # Greedy: argmax
        current = int(np.argmax(transitions[current]))
        tokens.append(current)
    return " ".join(i2w[t] for t in tokens)


def sample_generate(start_token="the", max_len=25):
    """Multinomial sampling: sample from the full distribution."""
    current = w2i[start_token]
    tokens = [current]
    for _ in range(max_len):
        if i2w[current] == "<EOS>":
            break
        probs = transitions[current]
        current = int(np.random.choice(vocab_size, p=probs))
        tokens.append(current)
    return " ".join(i2w[t] for t in tokens)


# Run comparisons
np.random.seed(42)

print("🔁 GREEDY SEARCH — Same output EVERY single time:")
print("=" * 60)
for i in range(4):
    result = greedy_generate("the")
    print(f"  Run {i+1}: {result}")

print()
print("🎲 SAMPLING — Different, more natural output each run:")
print("=" * 60)
for i in range(4):
    result = sample_generate("the")
    print(f"  Run {i+1}: {result}")

print()
print("=" * 60)
print("💡 Notice:")
print("  Greedy → 'the cat sat on the cat sat on the cat...' (stuck in a loop!)")
print("  Sampling → Different, more varied sentences each time")
print("  Real LLMs have this same problem, just at vocabulary scale (100K tokens)")

## 🎰 Multinomial Sampling — Random but Structured

### ELI12: Spinning a Weighted Wheel 🎡

Imagine a spinning wheel where each word gets a slice proportional to its probability.
- "cat" gets a big slice (40% likely) → big wedge
- "skateboard" gets a tiny slice (0.001% likely) → tiny wedge
- We spin the wheel and pick wherever it lands!

This breaks greedy's repetition loops. But there's a problem: **sometimes the wheel lands on "skateboard" even when talking about pets**, which sounds incoherent.

### Staff-Level: Multinomial Sampling

```
Algorithm:
  1. Run the LLM forward pass → get logits z ∈ ℝ^V
  2. Softmax: P(w) = exp(z_w) / Σ exp(z_i)  for all w in vocab
  3. Sample: w_t ~ Categorical(P)

Properties:
  ✅ Breaks repetition loops
  ✅ Diverse outputs on repeated queries
  ✅ Preserves distribution shape (high-prob tokens still favored)
  ❌ May sample very low-probability (incoherent) tokens
  ❌ For peaked distributions, top tokens dominate anyway
  ❌ For flat distributions, too much noise

The fix: truncate the distribution before sampling!
This is what top-k and top-p do.
```

In [ ]:
# ============================================================
# Multinomial Sampling with PyTorch
# ============================================================

torch.manual_seed(42)

# Simulated logits for the next token (small vocabulary for clarity)
# Imagine these are the output of a transformer for the prompt "The cat"
small_vocab = ["sat", "ate", "slept", "ran", "flew", "danced", "moonwalked", "teleported"]
logits = torch.tensor([3.5, 2.8, 2.2, 1.5, 0.8, 0.2, -0.5, -1.2])


def multinomial_sample(logits, vocab, n_samples=1000):
    """
    Sample from the full multinomial distribution.
    
    Steps:
      1. Softmax logits → probabilities
      2. torch.multinomial → sample indices
      3. Return sampled tokens
    """
    probs = F.softmax(logits, dim=0)
    # torch.multinomial draws samples from a multinomial distribution
    # replacement=True: each draw is independent
    sampled_indices = torch.multinomial(probs, num_samples=n_samples, replacement=True)
    return probs, sampled_indices


probs, samples = multinomial_sample(logits, small_vocab)

# Count how often each token was sampled
counts = torch.bincount(samples, minlength=len(small_vocab))
empirical_probs = counts.float() / counts.sum()

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

x = np.arange(len(small_vocab))
width = 0.35

# True probabilities vs empirical
bars1 = axes[0].bar(x - width/2, probs.numpy(), width, label='True P(token)',
                     color='#3498db', alpha=0.85, edgecolor='white')
bars2 = axes[0].bar(x + width/2, empirical_probs.numpy(), width, label='Sampled frequency',
                     color='#e74c3c', alpha=0.85, edgecolor='white')

axes[0].set_xticks(x)
axes[0].set_xticklabels([f'"The cat\n{w}"' for w in small_vocab], fontsize=9)
axes[0].set_ylabel('Probability / Frequency')
axes[0].set_title('🎰 Multinomial Sampling\n(True Distribution vs 1000 Samples)', fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(axis='y', alpha=0.3)

for bar in bars1:
    if bar.get_height() > 0.02:
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                     f'{bar.get_height():.2f}', ha='center', fontsize=8)

# Pie chart of sample distribution
threshold = 0.005
pie_probs = probs.numpy()
pie_labels = small_vocab
axes[1].pie(pie_probs, labels=[f'"{w}"\n{p:.1%}' for w, p in zip(pie_labels, pie_probs)],
            colors=plt.cm.Set3(np.linspace(0, 1, len(small_vocab))),
            autopct='', startangle=90, pctdistance=0.85)
axes[1].set_title('🎡 The Weighted Spinning Wheel\n("The cat ___")', fontweight='bold')

plt.tight_layout()
plt.show()

print()
print("Token probabilities after softmax:")
print("-" * 50)
for token, p in zip(small_vocab, probs.numpy()):
    bar = "█" * int(p * 50)
    print(f'  "The cat {token:>12}" : {p:.4f}  {bar}')

print()
print("💡 Even 'teleported' (0.4% prob) gets sampled occasionally!")
print(f"   'teleported' sampled {(samples == 7).sum().item()} times out of 1000")
print("   This is the incoherence problem with pure multinomial sampling.")

## 🔝 Top-k Sampling — Only Sample From the Best

### ELI12: The Top 5 Playlist 🎵

You want to listen to music but you have 50,000 songs. Instead of picking from ALL 50,000 (including that one terrible track from 2003), you:
1. Pick your **top 5 favorites**
2. Randomly pick from just those 5

That's top-k! It keeps randomness (variety!) but eliminates the terrible options.

### Staff-Level: Top-k Algorithm

```
Algorithm:
  1. Compute logits → probabilities P over vocabulary V
  2. Keep only the k tokens with highest probability
  3. Re-normalize those k probabilities to sum to 1
  4. Sample from this truncated distribution

Formally:
  V_top_k = {w : P(w) is in top-k of all P(v), v ∈ V}
  P'(w) = P(w) / Σ_{v ∈ V_top_k} P(v)  if w ∈ V_top_k, else 0
  w_t ~ Categorical(P')

Key tradeoff:
  k=1        → greedy decoding (deterministic)
  k=5        → conservative, stays very safe
  k=50       → more creative, more varied
  k=|V|      → full multinomial sampling

Problem with fixed k:
  For a PEAKED distribution (one token dominates):
    P = [0.90, 0.05, 0.02, 0.01, 0.01, 0.001, ...]
    k=50 still samples from 50 tokens even though only 1-2 matter!
    → Introduces unnecessary randomness

  For a FLAT distribution (all tokens similar prob):
    P = [0.01, 0.01, 0.01, ..., 0.01]  (100 tokens at 1% each)
    k=50 only covers 50% of the distribution
    → Too restrictive

This is the motivation for top-p (nucleus) sampling!
```

In [ ]:
# ============================================================
# Implement Top-k Sampling from Scratch
# ============================================================

def top_k_sampling(logits, k, temperature=1.0):
    """
    Top-k sampling from scratch.
    
    Args:
        logits: Raw model output, shape (vocab_size,)
        k: Number of top tokens to keep
        temperature: Divide logits by T before softmax (see later cell)
    
    Returns:
        sampled_index: The index of the sampled token
        probs_after_topk: The distribution after top-k filtering
        mask: Boolean mask showing which tokens are kept
    """
    # Apply temperature
    scaled_logits = logits / temperature
    
    # Step 1: Find the k-th highest logit value
    # torch.topk returns (values, indices) of the top-k elements
    top_k_values, top_k_indices = torch.topk(scaled_logits, k)
    
    # Step 2: Set all other logits to -infinity (they'll become 0 after softmax)
    filtered_logits = torch.full_like(scaled_logits, float('-inf'))
    filtered_logits[top_k_indices] = top_k_values
    
    # Step 3: Softmax over remaining tokens (renormalize)
    probs = F.softmax(filtered_logits, dim=0)
    
    # Step 4: Sample from truncated distribution
    sampled_index = torch.multinomial(probs, num_samples=1).item()
    
    # Boolean mask for visualization
    mask = filtered_logits != float('-inf')
    
    return sampled_index, probs, mask


# Use a larger vocabulary to show the effect clearly
torch.manual_seed(42)
np.random.seed(42)

# Create a vocabulary of 20 tokens with exponentially decaying probabilities
demo_vocab = [f"token_{i}" for i in range(20)]
raw_logits = torch.tensor([4.0, 3.5, 3.0, 2.5, 2.0, 1.5, 1.2, 1.0, 0.8, 0.6,
                            0.4, 0.2, 0.0, -0.2, -0.4, -0.6, -0.8, -1.0, -1.5, -2.0])
full_probs = F.softmax(raw_logits, dim=0)

k5_idx, k5_probs, k5_mask = top_k_sampling(raw_logits, k=5)
k15_idx, k15_probs, k15_mask = top_k_sampling(raw_logits, k=15)

# Visualize the effect of different k values
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

x = np.arange(len(demo_vocab))
colors_full = ['#3498db'] * len(demo_vocab)
colors_k5 = ['#2ecc71' if m else '#ecf0f1' for m in k5_mask.numpy()]
colors_k15 = ['#e74c3c' if m else '#ecf0f1' for m in k15_mask.numpy()]

# Full distribution
axes[0].bar(x, full_probs.numpy(), color=colors_full, edgecolor='white')
axes[0].set_title('📊 Full Distribution\n(Pure multinomial sampling)', fontweight='bold')
axes[0].set_xlabel('Token Index')
axes[0].set_ylabel('Probability')
axes[0].grid(axis='y', alpha=0.3)
axes[0].text(10, max(full_probs.numpy()) * 0.7,
             'All tokens eligible\n(even very low-prob ones)',
             fontsize=9, color='#666', bbox=dict(facecolor='white', alpha=0.7))

# k=5
axes[1].bar(x, k5_probs.numpy(), color=colors_k5, edgecolor='white')
axes[1].set_title('🔝 Top-k (k=5)\n(Conservative — only top 5)', fontweight='bold')
axes[1].set_xlabel('Token Index')
axes[1].set_ylabel('Renormalized Probability')
axes[1].grid(axis='y', alpha=0.3)
axes[1].text(6, max(k5_probs.numpy()) * 0.7,
             'Only 5 tokens eligible\n(gray = zeroed out)',
             fontsize=9, color='#666', bbox=dict(facecolor='white', alpha=0.7))

# k=15
axes[2].bar(x, k15_probs.numpy(), color=colors_k15, edgecolor='white')
axes[2].set_title('🔝 Top-k (k=15)\n(Creative — top 15 tokens)', fontweight='bold')
axes[2].set_xlabel('Token Index')
axes[2].set_ylabel('Renormalized Probability')
axes[2].grid(axis='y', alpha=0.3)
axes[2].text(0, max(k15_probs.numpy()) * 0.5,
             'Top 15 eligible\n(still cuts low-quality tail)',
             fontsize=9, color='#666', bbox=dict(facecolor='white', alpha=0.7))

plt.suptitle('🔝 Top-k Sampling: Trimming the Probability Distribution', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Show sampling statistics
print("\n" + "=" * 60)
print("Top-k Sampling Statistics:")
print("=" * 60)

for k_val in [5, 15, 20]:
    _, _, mask = top_k_sampling(raw_logits, k=k_val)
    covered_prob = F.softmax(raw_logits, dim=0)[mask].sum().item()
    print(f"  k={k_val:2d}: keeps {k_val:2d}/{len(demo_vocab)} tokens, "
          f"covers {covered_prob:.1%} of probability mass")

print()
print("💡 Key insight: k=5 covers most of the probability mass")
print("   because this is a peaked distribution.")
print("   But k=5 would be too restrictive for a flat distribution!")

## 🌐 Top-p (Nucleus) Sampling — Dynamically Smart

### ELI12: Pizza Toppings 🍕

You're ordering pizza for a party. Instead of picking "top 5 toppings always", you say:
> *"Keep adding toppings until we've covered 90% of what people like."*

If everyone loves just cheese and pepperoni (peaked distribution), you stop after 2 toppings.
If everyone has different favorites (flat distribution), you might need 15 toppings.

**Top-p adapts to the distribution shape.** That's the genius.

### Staff-Level: Nucleus Sampling Algorithm

```
Algorithm (Holtzman et al. 2020):
  1. Sort tokens by probability (descending)
  2. Compute cumulative sum of sorted probabilities
  3. Keep the smallest set of tokens whose cumulative prob ≥ p
     (this is the "nucleus")
  4. Re-normalize and sample from nucleus

Formally:
  Sort: p(w_1) ≥ p(w_2) ≥ ... ≥ p(w_|V|)
  Find: smallest m such that Σᵢ₌₁ᵐ p(wᵢ) ≥ p
  Nucleus: V_top = {w_1, ..., w_m}
  Sample: w_t ~ Categorical(P restricted to V_top)

Why it beats top-k:

  Peaked distribution (e.g., factual Q&A):
    P = [0.90, 0.05, 0.03, 0.01, 0.005, ...]
    Top-k=50 → still samples 50 tokens (includes garbage)
    Top-p=0.9 → only samples 1 token (just the dominant one!)

  Flat distribution (e.g., creative writing):
    P = [0.02, 0.02, 0.02, ..., 0.02]  (50 tokens)
    Top-k=5 → only covers 10% of valid options (too restrictive)
    Top-p=0.9 → keeps 45 tokens (preserves appropriate diversity)

Typical settings in practice:
  p=0.9 → standard creative generation
  p=0.95 → more creative
  p=1.0 → full multinomial sampling
```

In [ ]:
# ============================================================
# Implement Top-p Sampling from Scratch
# Compare k=50 vs p=0.9 on peaked and flat distributions
# ============================================================

def top_p_sampling(logits, p, temperature=1.0):
    """
    Top-p (nucleus) sampling from scratch.
    
    Args:
        logits: Raw model output, shape (vocab_size,)
        p: Cumulative probability threshold (e.g., 0.9)
        temperature: Scale logits before softmax
    
    Returns:
        sampled_index: Original vocabulary index of sampled token
        probs_filtered: Probability distribution after top-p filtering
        nucleus_size: How many tokens were in the nucleus
    """
    # Apply temperature
    scaled_logits = logits / temperature
    
    # Step 1: Compute probabilities
    probs = F.softmax(scaled_logits, dim=0)
    
    # Step 2: Sort probabilities in descending order
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    
    # Step 3: Compute cumulative sum
    cumulative_probs = torch.cumsum(sorted_probs, dim=0)
    
    # Step 4: Find nucleus — keep tokens until cumulative prob reaches p
    # We use a SHIFT trick: shift cumsum right by 1 so the token that
    # CROSSES the threshold is still included
    # Remove tokens where cumulative sum ALREADY exceeds p BEFORE this token
    to_remove = cumulative_probs - sorted_probs > p  # True = remove
    
    # Step 5: Zero out non-nucleus tokens
    sorted_probs_filtered = sorted_probs.clone()
    sorted_probs_filtered[to_remove] = 0.0
    
    # Step 6: Renormalize
    sorted_probs_filtered = sorted_probs_filtered / sorted_probs_filtered.sum()
    
    # Step 7: Sample from sorted distribution
    sampled_sorted_idx = torch.multinomial(sorted_probs_filtered, num_samples=1).item()
    sampled_index = sorted_indices[sampled_sorted_idx].item()
    
    # Re-map back to original order for visualization
    probs_filtered = torch.zeros_like(probs)
    nucleus_indices = sorted_indices[~to_remove]
    probs_filtered[nucleus_indices] = sorted_probs_filtered[~to_remove]
    
    nucleus_size = (~to_remove).sum().item()
    return sampled_index, probs_filtered, nucleus_size


# ─── Compare k=50 vs p=0.9 on two distribution types ───
VOCAB_SIZE = 100

# Peaked distribution: one dominant token (factual context)
peaked_logits = torch.zeros(VOCAB_SIZE)
peaked_logits[0] = 5.0   # Token 0 is VERY likely
peaked_logits[1] = 2.0
peaked_logits[2] = 1.0
peaked_logits[3:] = torch.linspace(0.5, -3.0, VOCAB_SIZE - 3)

# Flat distribution: many equally likely tokens (creative context)
flat_logits = torch.zeros(VOCAB_SIZE)
flat_logits[:50] = torch.linspace(1.0, 0.5, 50)  # 50 roughly equal tokens
flat_logits[50:] = torch.linspace(0.0, -3.0, 50)  # then declining

fig, axes = plt.subplots(2, 3, figsize=(22, 12))
fig.suptitle('🌐 Top-k vs Top-p: Why Nucleus Sampling Adapts Better', fontsize=16, fontweight='bold')

configs = [
    (peaked_logits, "PEAKED Distribution\n(Factual Q&A)"),
    (flat_logits,   "FLAT Distribution\n(Creative Writing)"),
]

for row, (logits_src, title) in enumerate(configs):
    full_p = F.softmax(logits_src, dim=0)
    
    # Top-k = 50
    _, topk_probs, topk_mask = top_k_sampling(logits_src, k=50)
    # Top-p = 0.9
    _, topp_probs, n_size = top_p_sampling(logits_src, p=0.9)
    nucleus_mask = topp_probs > 0
    
    topk_n = (topk_mask).sum().item()
    topp_n = n_size
    
    x = np.arange(VOCAB_SIZE)
    
    # Column 0: Full distribution
    axes[row, 0].bar(x, full_p.numpy(), color='#95a5a6', width=1.0, edgecolor='none')
    axes[row, 0].set_title(f'{title}\n(Full Distribution)', fontweight='bold', fontsize=11)
    axes[row, 0].set_xlabel('Token rank')
    axes[row, 0].set_ylabel('P(token)')
    axes[row, 0].grid(axis='y', alpha=0.3)
    
    # Column 1: Top-k=50
    colors_k = ['#e74c3c' if m else '#ecf0f1' for m in topk_mask.numpy()]
    axes[row, 1].bar(x, topk_probs.numpy(), color=colors_k, width=1.0, edgecolor='none')
    axes[row, 1].set_title(f'Top-k (k=50)\nKeeps {topk_n} tokens', fontweight='bold', fontsize=11,
                            color='#e74c3c')
    axes[row, 1].set_xlabel('Token rank')
    axes[row, 1].set_ylabel('Renorm. P(token)')
    axes[row, 1].grid(axis='y', alpha=0.3)
    
    # Add annotation for peaked case
    if row == 0:
        axes[row, 1].annotate('⚠️ Still sampling\n48 junk tokens!',
                               xy=(25, topk_probs[25].item() + 0.003),
                               fontsize=9, color='#e74c3c', ha='center',
                               bbox=dict(facecolor='white', alpha=0.8, boxstyle='round'))
    else:
        axes[row, 1].annotate('⚠️ Only covers\ntop 50 of ~50 valid tokens',
                               xy=(25, max(topk_probs.numpy()) * 0.7),
                               fontsize=9, color='#e74c3c', ha='center',
                               bbox=dict(facecolor='white', alpha=0.8, boxstyle='round'))
    
    # Column 2: Top-p=0.9
    colors_p = ['#2ecc71' if m else '#ecf0f1' for m in nucleus_mask.numpy()]
    axes[row, 2].bar(x, topp_probs.numpy(), color=colors_p, width=1.0, edgecolor='none')
    axes[row, 2].set_title(f'Top-p (p=0.9)\nKeeps {topp_n} tokens (adaptive!)', fontweight='bold',
                            fontsize=11, color='#27ae60')
    axes[row, 2].set_xlabel('Token rank')
    axes[row, 2].set_ylabel('Renorm. P(token)')
    axes[row, 2].grid(axis='y', alpha=0.3)
    
    if row == 0:
        axes[row, 2].annotate(f'✅ Only {topp_n} token(s)\n(correctly focused!)',
                               xy=(topp_n + 2, max(topp_probs.numpy()) * 0.5),
                               fontsize=9, color='#27ae60', ha='left',
                               bbox=dict(facecolor='white', alpha=0.8, boxstyle='round'))
    else:
        axes[row, 2].annotate(f'✅ {topp_n} tokens\n(captures diversity!)',
                               xy=(topp_n * 0.5, max(topp_probs.numpy()) * 0.7),
                               fontsize=9, color='#27ae60', ha='center',
                               bbox=dict(facecolor='white', alpha=0.8, boxstyle='round'))

plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("The Nucleus Sampling Advantage:".upper())
print("=" * 70)
for logits_src, label in configs:
    _, _, topk_mask_v = top_k_sampling(logits_src, k=50)
    _, _, n_sz = top_p_sampling(logits_src, p=0.9)
    print(f"  {label.replace(chr(10), ' '):40s}  top-k=50 → {topk_mask_v.sum().item()} tokens  |  top-p=0.9 → {n_sz} tokens")

In [ ]:
# ============================================================
# Temperature: The Spice Dial 🌡️
# ============================================================
# Temperature T scales logits BEFORE softmax:
#   P(w) = exp(z_w / T) / Σ exp(z_i / T)
#
# Low T  → peaked distribution → model is confident, conservative
# High T → flat distribution   → model is random, creative

torch.manual_seed(42)

# Base logits (before temperature)
base_logits = torch.tensor([3.0, 2.0, 1.5, 1.0, 0.5, 0.0, -0.5, -1.0])
token_names = ["Paris", "London", "Berlin", "Rome", "Tokyo",
               "soccer", "banana", "galaxy"]

temperatures = [0.1, 0.5, 1.0, 2.0]
temp_labels = [
    "T=0.1\n(Very Cold 🥶)\nNearly greedy",
    "T=0.5\n(Cool 😊)\nConservative",
    "T=1.0\n(Normal 😐)\nDefault",
    "T=2.0\n(Hot 🔥)\nVery creative",
]
colors = ['#2980b9', '#27ae60', '#f39c12', '#e74c3c']

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
fig.suptitle('🌡️ Temperature Parameter: The Creativity Dial\n'
             '(Prompt: "The capital of France is ___")',
             fontsize=15, fontweight='bold')

for ax, T, label, color in zip(axes, temperatures, temp_labels, colors):
    probs = F.softmax(base_logits / T, dim=0).numpy()
    
    bars = ax.bar(range(len(token_names)), probs, color=color, alpha=0.85, edgecolor='white', linewidth=1.5)
    ax.set_xticks(range(len(token_names)))
    ax.set_xticklabels(token_names, rotation=45, ha='right', fontsize=10)
    ax.set_ylabel('P(token)')
    ax.set_ylim(0, 1.05)
    ax.set_title(label, fontweight='bold', color=color, fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    
    # Label the bars
    for bar, p in zip(bars, probs):
        if p > 0.015:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{p:.2f}', ha='center', fontsize=8, fontweight='bold')
    
    # Entropy annotation
    entropy = -np.sum(probs * np.log(probs + 1e-10))
    ax.text(0.5, 0.92, f'Entropy: {entropy:.2f} bits',
            transform=ax.transAxes, ha='center', fontsize=9,
            bbox=dict(facecolor='white', edgecolor=color, boxstyle='round', linewidth=2))

plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("🌡️ TEMPERATURE EFFECTS SUMMARY")
print("=" * 70)

for T in temperatures:
    probs = F.softmax(base_logits / T, dim=0).numpy()
    entropy = -np.sum(probs * np.log(probs + 1e-10))
    top1_conf = probs[0]
    print(f"  T={T:.1f}: P('Paris')={top1_conf:.4f},  Entropy={entropy:.3f} bits")

print()
print("""
When to use each temperature:

  T < 0.7  : Factual Q&A, math, code generation
             → "Be precise, not creative"

  T = 1.0  : Conversational chatbot (default ChatGPT)
             → Balanced. Feels natural.

  T = 1.2  : Creative writing, brainstorming
             → More varied, sometimes surprising

  T > 1.5  : Experimental / poetry / very random
             → Output can become incoherent quickly!
""")

## ⛔ Repetition Penalty — Stopping the Parrot

### ELI12: The "Stop Repeating Yourself!" Rule 🗣️

Even with top-p sampling and temperature, language models can get into a rut:
> *"I think AI is great. I think AI is great. I think AI is really, really great. AI is great."*

The repetition penalty is simple: **if you've already said a word, make it less likely to say it again.**

### Staff-Level: How It Works

```
Algorithm (Keskar et al., 2019 — CTRL paper):

  For each token w in the vocabulary:
    If w has appeared in the current context:
      If logit[w] > 0:  logit[w] = logit[w] / θ
      Else:             logit[w] = logit[w] * θ
    (where θ > 1.0 is the repetition penalty)

  This REDUCES the logit of already-seen tokens.

Intuition:
  θ = 1.0 → no penalty (standard sampling)
  θ = 1.3 → mild penalty (common in practice)
  θ = 1.5 → strong penalty
  θ = 2.0 → very aggressive (may hurt quality)

Tradeoff:
  Too low (θ ≈ 1.0): repetition occurs
  Too high (θ >> 1.0): model avoids correct words it needs to repeat
    (e.g., "the" might need to appear many times legitimately!)

Practical note:
  Often combined with a sliding window — only penalize tokens
  seen in the last N tokens, not the entire context.
  (Names, technical terms should be repeatable if they're
   the topic of discussion!)
```

ChatGPT and modern LLMs use repetition penalty by default, usually with θ ≈ 1.1–1.3.

## 📊 Evaluation Deep Dive — How Do We Know It's Good?

### ELI12: Report Cards for AI 📝

You wouldn't trust a student just because they *say* they're smart. You'd give them a test!
For language models, the "report cards" are called **benchmarks**.

Different benchmarks test different skills, just like different school subjects:

| Benchmark | The "Subject" | Passing grade |
|-----------|--------------|---------------|
| **MMLU** | General knowledge (57 subjects) | >90% = elite model |
| **HumanEval** | Coding (write real Python) | >80% = impressive |
| **GSM8K** | Math word problems | >90% = strong reasoning |
| **TruthfulQA** | Honesty (don't make things up) | >70% = trustworthy |
| **HellaSwag** | Common sense | >95% = human-level |

### Staff-Level: What Makes a Good Benchmark?

```
Requirements for a useful benchmark:

1. TASK-APPROPRIATE FORMAT
   Multiple-choice (MMLU, HellaSwag): easy to evaluate, but narrow
   Open-ended generation (TruthfulQA): harder to evaluate, more realistic
   Code execution (HumanEval): gold standard — run the code!

2. CONTAMINATION-FREE
   Training data must NOT contain benchmark answers
   This is a major problem in practice — many benchmarks are "leaked"
   GPT-4 technical report spends significant time on this!

3. DIFFICULT ENOUGH
   If every model scores 95%+, the benchmark is saturated and useless
   MMLU was hard in 2021, saturated by 2024 for frontier models

4. REPRESENTATIVE
   Should test the distribution of real user tasks
   LMSYS Chatbot Arena (ELO ratings) is closest to this goal

5. REPRODUCIBLE
   Same model, same benchmark = same score (different from creative tasks)
```

In [ ]:
# ============================================================
# MMLU-Style Questions: What Does the Benchmark Actually Test?
# ============================================================

print("📊 MMLU-STYLE BENCHMARK QUESTIONS")
print("=" * 70)
print("MMLU = Massive Multitask Language Understanding")
print("57 subjects × multiple-choice questions")
print()

mmlu_examples = [
    {
        "subject": "🧮 STEM — College Mathematics",
        "question": "What is the derivative of f(x) = x³ · sin(x)?",
        "choices": [
            "A) 3x² · sin(x)",
            "B) x³ · cos(x)",
            "C) 3x² · sin(x) + x³ · cos(x)",
            "D) 3x² · cos(x) + x³ · sin(x)"
        ],
        "answer": "C",
        "what_it_tests": "Product rule: (f·g)' = f'·g + f·g'. Tests whether the model can apply",
        "detail": "multi-rule calculus and not just pattern-match to a single rule."
    },
    {
        "subject": "🖥️ STEM — Computer Science",
        "question": "What is the time complexity of finding the k-th smallest element\n"
                    "   in an unsorted array using the QuickSelect algorithm (average case)?",
        "choices": [
            "A) O(n log n)",
            "B) O(n)          ← average case",
            "C) O(n²)         ← worst case",
            "D) O(log n)"
        ],
        "answer": "B",
        "what_it_tests": "Algorithm analysis. Tests whether the model knows QuickSelect is O(n) average",
        "detail": "(same partition logic as QuickSort) vs O(n²) worst case. Subtle but important!"
    },
    {
        "subject": "🌍 Social Sciences — World History",
        "question": "The Bretton Woods system established in 1944 primarily created which institution?",
        "choices": [
            "A) The United Nations",
            "B) The World Bank and IMF",
            "C) The World Trade Organization",
            "D) NATO"
        ],
        "answer": "B",
        "what_it_tests": "Factual recall + context. Tests whether the model has absorbed historical",
        "detail": "economic facts (not just famous battles). MMLU covers ALL of human knowledge!"
    },
    {
        "subject": "🩺 STEM — Medical — Clinical Knowledge",
        "question": "A 45-year-old presents with fatigue, pallor, and macrocytic anemia.\n"
                    "   Serum B12 is low. Which test best confirms pernicious anemia?",
        "choices": [
            "A) Serum folate level",
            "B) Intrinsic factor antibodies",
            "C) Bone marrow biopsy",
            "D) Peripheral blood smear"
        ],
        "answer": "B",
        "what_it_tests": "Clinical reasoning under uncertainty. Tests multi-step reasoning:",
        "detail": "B12 deficiency → causes → pernicious anemia mechanism → diagnostic test. Safety-critical!"
    },
    {
        "subject": "💡 Common Sense — Logical Reasoning",
        "question": "If all glorks are blips, and some blips are zaps, which must be true?",
        "choices": [
            "A) All glorks are zaps",
            "B) Some glorks are zaps",
            "C) No glorks are zaps",
            "D) None of the above can be concluded"
        ],
        "answer": "D",
        "what_it_tests": "Logical reasoning with novel objects (glorks/blips = no prior knowledge!).",
        "detail": "Tests pure deductive logic, not world knowledge. D is correct: some blips are\n"
                  "   zaps but none of those overlap with glorks, so we CANNOT conclude B."
    },
]

for i, q in enumerate(mmlu_examples, 1):
    print(f"\n{'─' * 70}")
    print(f"  Question {i} — {q['subject']}")
    print(f"{'─' * 70}")
    print(f"  Q: {q['question']}")
    print()
    for choice in q['choices']:
        marker = " ← CORRECT" if choice.startswith(q['answer']) else ""
        print(f"     {choice}{marker}")
    print()
    print(f"  🎯 What it tests: {q['what_it_tests']}")
    print(f"                   {q['detail']}")

print()
print("=" * 70)
print("\n📈 MMLU SCORES BY MODEL (rough figures as of early 2024):")
print("-" * 50)
scores = [
    ("Random baseline", 25.0),
    ("GPT-3 (175B)", 43.9),
    ("Llama 2 13B", 54.8),
    ("Llama 2 70B", 68.9),
    ("GPT-3.5 Turbo", 70.0),
    ("GPT-4", 86.4),
    ("Claude 3 Opus", 88.2),
    ("Human expert", 89.8),
]
for model, score in scores:
    bar = "█" * int(score / 3)
    print(f"  {model:20s}: {score:5.1f}%  {bar}")

## 🛡️ Safety Evaluation — The Hard Problems

### ELI12: Testing for Bad Behavior 🚨

A chatbot that's smart but MEAN is still a terrible chatbot. Safety evaluation is like testing whether the AI:
1. **Stays polite and helpful** (not toxic)
2. **Treats everyone fairly** (no bias)
3. **Tells the truth** (not hallucinating)
4. **Can't be tricked** into doing bad things (adversarial robustness)

### Staff-Level: Safety Benchmarks

| Category | Benchmark | What It Measures | Key Metric |
|----------|-----------|-----------------|------------|
| **Toxicity** | RealToxicityPrompts | P(toxic continuation \| neutral prompt) | Toxicity rate % |
| **Bias** | WinoBias, BBQ | Occupation/gender/race association | Bias score |
| **Truthfulness** | TruthfulQA | Does the model believe false things? | % true AND informative |
| **Harmlessness** | AdvBench, HarmBench | Jailbreak success rate | Attack success rate % |
| **Hallucination** | FActScorer, HaluEval | Factual precision in long-form gen. | Factual precision % |
| **Calibration** | Various | P(model says it knows) vs. actual accuracy | Expected Calibration Error |

### The Alignment Tax

```
A dirty secret of safety evaluation:

  Safety filtering often REDUCES capability on some tasks.

  Example: Refusal rate on gray-area requests:
    GPT-4 with strong safety filters: refuses writing violent fiction = "safe"
    But also refuses security research = "over-refusal" problem

  The challenge: calibrate the safety system to refuse genuinely
  harmful requests while still being HELPFUL for legitimate ones.
  Getting this balance right is an open research problem.

Key metrics OpenAI tracks:
  - False positive rate (FPR): Refusing safe requests ← bad for UX
  - False negative rate (FNR): Allowing harmful requests ← bad for safety
  - Goal: Minimize both simultaneously (classic precision-recall tradeoff!)
```

In [ ]:
# ============================================================
# Simple Bias Detection: Probing for Gender Bias in Occupations
# ============================================================
# This is a simplified version of the WinoBias / coreference bias
# evaluation technique.
#
# REAL technique: feed templates to an LLM, measure log-probability
# of gendered pronouns (he/she/they) for different occupations.
# Here we simulate what such a study might find.

print("🔍 GENDER BIAS IN LLM OCCUPATION ASSOCIATIONS")
print("=" * 70)
print("Template: 'The {occupation} said that ___ was tired.'")
print("Measuring: P('he') vs P('she') as next token")
print()

# Simulated results from a bias probe
# (Based on trends reported in real bias studies)
# Format: (occupation, P_he, P_she, P_they, stereotyped_direction)
bias_data = [
    # occupation          P(he)  P(she) P(they)  census_reality
    ("nurse",             0.22,  0.68,  0.10,    "F-dominated (~88% female)"),
    ("doctor",            0.58,  0.32,  0.10,    "Mixed (~40% female)"),
    ("engineer",          0.70,  0.18,  0.12,    "M-dominated (~15% female)"),
    ("teacher",           0.30,  0.58,  0.12,    "F-dominated (~74% female)"),
    ("CEO",               0.74,  0.18,  0.08,    "M-dominated (~8% female)"),
    ("secretary",         0.14,  0.76,  0.10,    "F-dominated (~93% female)"),
    ("programmer",        0.72,  0.18,  0.10,    "M-dominated (~20% female)"),
    ("psychologist",      0.32,  0.56,  0.12,    "F-dominated (~77% female)"),
]

occupations = [d[0] for d in bias_data]
p_he = np.array([d[1] for d in bias_data])
p_she = np.array([d[2] for d in bias_data])
p_they = np.array([d[3] for d in bias_data])
census_info = [d[4] for d in bias_data]

# Define a simple bias score: |P(he) - P(she)| normalized
# Higher = more biased toward one gender
bias_score = np.abs(p_he - p_she)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('🛡️ Probing Language Models for Gender Bias in Occupations\n'
             '(Simulated pronoun prediction probabilities)', fontsize=14, fontweight='bold')

# Stacked bar chart: P(he) vs P(she) vs P(they)
x = np.arange(len(occupations))
width = 0.6

bars1 = axes[0].barh(x, p_he, width, label='P("he")', color='#3498db', alpha=0.85)
bars2 = axes[0].barh(x, p_she, width, left=p_he, label='P("she")', color='#e91e8c', alpha=0.85)
bars3 = axes[0].barh(x, p_they, width, left=p_he+p_she, label='P("they")', color='#95a5a6', alpha=0.85)

axes[0].set_yticks(x)
axes[0].set_yticklabels([f'The {occ}...' for occ in occupations], fontsize=11)
axes[0].set_xlabel('Probability of Next Pronoun')
axes[0].set_title('Pronoun Distribution per Occupation', fontweight='bold')
axes[0].legend(loc='lower right', fontsize=10)
axes[0].axvline(x=0.5, color='black', linestyle='--', alpha=0.4, linewidth=1)
axes[0].text(0.5, -0.7, 'Equal →', ha='center', fontsize=9, color='gray')
axes[0].grid(axis='x', alpha=0.3)

# Add P(he) labels
for i, (ph, ps) in enumerate(zip(p_he, p_she)):
    axes[0].text(ph/2, i, f'{ph:.0%}', ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    axes[0].text(ph + ps/2, i, f'{ps:.0%}', ha='center', va='center', fontsize=8, color='white', fontweight='bold')

# Bias score chart
bar_colors = ['#e74c3c' if b > 0.3 else '#f39c12' if b > 0.15 else '#2ecc71' for b in bias_score]
axes[1].barh(x, bias_score, color=bar_colors, alpha=0.85, edgecolor='white')
axes[1].set_yticks(x)
axes[1].set_yticklabels(occupations, fontsize=11)
axes[1].set_xlabel('Bias Score = |P(he) - P(she)|')
axes[1].set_title('Bias Score per Occupation\n(Higher = More Biased)', fontweight='bold')
axes[1].axvline(x=0.3, color='#e74c3c', linestyle='--', alpha=0.6, linewidth=2)
axes[1].text(0.31, -0.5, 'High bias\nthreshold', fontsize=9, color='#e74c3c')
axes[1].grid(axis='x', alpha=0.3)

for i, (score, census) in enumerate(zip(bias_score, census_info)):
    axes[1].text(score + 0.01, i, census, va='center', fontsize=8, color='#555')

# Legend
patches = [
    mpatches.Patch(color='#e74c3c', label='High bias (>0.3)'),
    mpatches.Patch(color='#f39c12', label='Medium bias (0.15-0.3)'),
    mpatches.Patch(color='#2ecc71', label='Low bias (<0.15)'),
]
axes[1].legend(handles=patches, fontsize=9, loc='lower right')

plt.tight_layout()
plt.show()

print()
print("=" * 70)
print("🔍 ANALYSIS: What This Bias Probe Reveals")
print("=" * 70)
print("""
1. REFLECTS TRAINING DATA BIASES:
   The model associates 'nurse'/'secretary' with she and 'engineer'/'CEO' with he.
   This mirrors historical gender distributions in text data — the internet
   was written by/about people in a biased world!

2. MOSTLY TRACKS REALITY... BUT REINFORCES IT:
   Some associations ARE statistically valid (88% of nurses are female).
   But the model strengthens stereotypes rather than offering neutral predictions.
   A 'doctor' should ideally predict 50/50 — the model says 58% he / 32% she.

3. HOW TO FIX IT:
   a) RLHF with debiasing reward signal (score responses on bias)
   b) Fine-tune on balanced/counterfactual data
   c) Post-processing: flag and neutralize biased outputs
   d) Counterfactual data augmentation (swap he→she in training data)

4. WHY THIS MATTERS IN PRACTICE:
   Resume screening, medical advice, loan applications — real-world AI
   deployments can amplify bias at scale. Evaluation is step 1.
""")

## 🏗️ ChatGPT System Design — The Full Pipeline

### ELI12: A Restaurant Kitchen Analogy 🍽️

When you order food at a restaurant:
1. **Host checks if you're welcome** (Input safety filter — no troublemakers!)
2. **Waiter takes your exact order** (Prompt enhancer — adds context, preferences)
3. **Chef cooks your food** (Response generator — the LLM)
4. **Food inspector checks the plate** (Output safety filter — no bad ingredients!)
5. **You get your meal** 🎉

ChatGPT works the same way. Your message goes through multiple stages before the response appears.

### Staff-Level: The Full Architecture

```
┌─────────────────────────────────────────────────────────────────────┐
│                    CHATGPT SYSTEM DESIGN                            │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   USER MESSAGE                                                      │
│        │                                                            │
│        ▼                                                            │
│   ┌─────────────────────────────┐                                   │
│   │  INPUT SAFETY FILTER       │  Classifier: toxic/jailbreak/pii? │
│   │  Latency: <5ms             │  Block → return canned response   │
│   └──────────────┬──────────────┘                                   │
│                  │ (safe input passes through)                     │
│                  ▼                                                  │
│   ┌─────────────────────────────┐  ◄── Session History DB         │
│   │  PROMPT ENHANCER           │       (Redis / DynamoDB)         │
│   │  Adds:                     │  ◄── System prompt template      │
│   │  • System prompt           │  ◄── User preferences            │
│   │  • Conversation history    │                                   │
│   │  • Context window mgmt     │  Manages token budget carefully!  │
│   └──────────────┬──────────────┘                                   │
│                  │                                                  │
│                  ▼                                                  │
│   ┌─────────────────────────────┐                                   │
│   │  RESPONSE GENERATOR (LLM)  │  GPU cluster, KV-cache           │
│   │  Sampling: top-p=0.9, T=1  │  Streaming: SSE / WebSocket      │
│   │  Repetition penalty: 1.1   │  Continuous batching             │
│   └──────────────┬──────────────┘                                   │
│                  │                                                  │
│                  ▼                                                  │
│   ┌─────────────────────────────┐                                   │
│   │  OUTPUT SAFETY FILTER      │  Catch: harmful content, PII     │
│   │  Latency: <10ms            │  Classifier OR reward model      │
│   └──────────────┬──────────────┘                                   │
│                  │                                                  │
│                  ▼                                                  │
│   RESPONSE TO USER (streamed token by token) 🎉                    │
│                                                                     │
│   MONITORING (async):                                               │
│   • Latency: p50 <500ms, p99 <2s first token                       │
│   • Toxicity rate (< 0.01% target)                                 │
│   • User thumbs up/down ratio                                      │
│   • Context window utilization                                      │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### Key Design Decisions at Scale

| Challenge | Solution | Why |
|-----------|----------|-----|
| **Multi-turn memory** | Session Manager stores last N turns | Context windows have limits; older history is summarized or dropped |
| **Fast inference** | KV-cache + continuous batching | Recomputing attention for every token would be O(n²) per step |
| **Streaming** | Server-Sent Events (SSE) | Users see tokens appear progressively → perceived latency drops |
| **Scale** | Tensor parallelism + model replicas | Single A100 holds ~7B model; 175B needs 8+ GPUs with tensor parallel |
| **Safety latency** | Lightweight classifiers for filter layers | The 7B safety classifier can't add 2s of latency to every request |
| **Long conversations** | Sliding window or summarization | At 128K context, attention cost = O(128K²) = expensive |


## 🎯 Interview Cheat Sheet — Sampling Deep Dive

### Top-k vs Top-p: When to Use Which

| Question | Answer |
|----------|--------|
| **"What is top-k sampling?"** | Keep only the k highest-probability tokens; re-normalize and sample. Fixed-size truncation. |
| **"What is top-p (nucleus) sampling?"** | Keep the smallest set of tokens whose cumulative probability ≥ p; re-normalize and sample. Adaptive-size truncation. |
| **"Why is top-p better than top-k?"** | Top-p adapts to the distribution shape. Peaked distribution → small nucleus (focused). Flat distribution → large nucleus (diverse). Top-k always keeps exactly k tokens regardless of distribution shape. |
| **"What temperature does ChatGPT use?"** | Approximately T=1.0 for conversational tasks, lower for factual/code tasks. Exact values are proprietary. |
| **"When would you use T < 1.0?"** | Code generation, math, factual Q&A — where you want the most likely correct answer |
| **"When would you use T > 1.0?"** | Creative writing, brainstorming, poetry — where diversity and surprise are valuable |
| **"What's the problem with pure multinomial sampling?"** | May sample very low-probability (incoherent/wrong) tokens. This is why we truncate with top-k or top-p. |
| **"What's beam search's problem for chat?"** | Text degeneration — repetitive loops. Natural language is not deterministic; beam search amplifies the safe/common path. |
| **"Why use repetition penalty?"** | Language models tend to loop on high-probability phrases. Repetition penalty reduces the logit of previously generated tokens. |

### Numbers to Know

| Setting | Typical Value | Use Case |
|---------|--------------|----------|
| temperature (chat) | 0.7 – 1.0 | Conversational ChatGPT |
| temperature (code) | 0.0 – 0.3 | Code generation, math |
| top-p | 0.9 – 0.95 | General generation |
| top-k | 40 – 100 | Alternative to top-p |
| repetition penalty | 1.1 – 1.3 | Reduce loops |
| MMLU human expert score | ~89.8% | Bar to beat |
| GPT-4 MMLU | ~86.4% | Close to human expert |

### The "Design ChatGPT" Sampling Answer (Interview Gold)

```
"For text generation, I would use top-p nucleus sampling (p=0.9) with
temperature (T=0.7-1.0 depending on task type) and a mild repetition
penalty (1.1). Here's why each choice:

  Top-p over top-k: Because top-p adapts to distribution shape.
  When the model is confident, the nucleus is small (focused).
  When uncertain, the nucleus is large (diverse). Fixed-k would
  over-constrain confident predictions and under-constrain
  uncertain ones.

  Temperature: Lower for factual tasks (code, math) to maximize
  probability of correct answer. Higher for creative tasks to
  increase diversity and avoid generic outputs.

  Repetition penalty: Small penalty prevents obvious looping without
  aggressively preventing legitimate word reuse (articles, names).

  For beam search: NEVER for open-ended chat. Beam search causes
  degenerate repetition in autoregressive generation for tasks
  with high output entropy. Reserve beam search for short,
  constrained outputs like translation."
```

## 🧠 Quiz — Test Your Knowledge!

Think about each question before revealing the answer.

---

**Question 1:** You're building a math tutoring chatbot. A student asks: *"What is 2+2?"*
Which sampling settings should you use, and why?

<details>
<summary>💡 Reveal Answer</summary>

**Use low temperature (T ≈ 0.1–0.3) with top-p=0.9 (or even greedy/top-k=1).**

Reasoning: For factual, deterministic questions like arithmetic, there is ONE correct answer. You want the model to be highly confident and pick the most probable (correct) token. High temperature would introduce randomness that could produce wrong answers (e.g., "5"). Low temperature = very peaked distribution = model almost always picks "4".

</details>

---

**Question 2:** You have a vocabulary of 50,000 tokens. After computing probabilities, the distribution looks like:
- Token A: 0.75
- Token B: 0.12
- Token C: 0.05
- All other 49,997 tokens: 0.08 combined

How many tokens does top-k=50 keep vs top-p=0.9?

<details>
<summary>💡 Reveal Answer</summary>

**Top-k=50 keeps exactly 50 tokens.**
**Top-p=0.9 keeps only 2 tokens (A + B = 0.87 ≥ 0.9... wait, 0.75+0.12=0.87 < 0.90, so we need C too = 0.92 ≥ 0.90 → keeps 3 tokens).**

Actually: cumsum after sorting = [0.75, 0.87, 0.92, ...]. We need cumsum ≥ 0.90, which is reached at token C (cumsum = 0.92). So top-p=0.9 keeps **3 tokens**.

Key takeaway: Top-k=50 includes 47 tokens with collectively only 8% probability. Top-p=0.9 is much smarter here — it focuses on the 3 meaningful options.

</details>

---

**Question 3:** What is "reward hacking" in RLHF, and how does the KL penalty prevent it?

<details>
<summary>💡 Reveal Answer</summary>

**Reward hacking:** The LLM learns to maximize the reward model's score through degenerate strategies that score high but produce low-quality output. Example: generating repetitive sycophantic text like "I'm so helpful, I'm the most helpful AI!" which a naive reward model might score highly.

**KL penalty:** The PPO objective includes `-β·KL(π_θ || π_SFT)` which penalizes the model for diverging too far from the SFT (reference) policy. This forces the model to stay in the "valid language" region of the output space, where reward hacking strategies don't exist. β controls the strength of this constraint — higher β = stays closer to SFT, harder to hack reward.

</details>

---

**Question 4:** You observe that your chatbot repeats phrases like "Certainly! I'd be happy to help!" at the start of EVERY response. What parameter would you adjust, and in which direction?

<details>
<summary>💡 Reveal Answer</summary>

Several options:

1. **Increase repetition penalty (θ):** e.g., from 1.0 to 1.3. This reduces the logit of tokens that appear frequently in generated text, making the model less likely to reuse "Certainly!"

2. **Increase temperature slightly:** Flattens the distribution, making the model less likely to always pick the highest-probability opener.

3. **Best fix: RLHF/DPO with human feedback:** If humans rate responses with sycophantic openers as worse, the model will learn to avoid them. This is more robust than hyperparameter tuning.

4. **System prompt instruction:** Tell the model in the system prompt: "Do not start responses with 'Certainly!' or similar phrases." This is the most reliable quick fix.

</details>

---

**Question 5:** A researcher says: "Our model scored 72% on MMLU, proving it's a great chatbot!" What's wrong with this claim?

<details>
<summary>💡 Reveal Answer</summary>

Multiple issues:

1. **MMLU ≠ chatbot quality.** MMLU tests static multiple-choice knowledge. Chatbot quality involves conversational coherence, instruction following, multi-turn context, and helpfulness — none of which MMLU measures.

2. **No safety evaluation.** 72% MMLU says nothing about toxicity, bias, hallucination rate, or adversarial robustness.

3. **Benchmark contamination risk.** Was the MMLU test data in the training set? A model can "score high" by memorizing answers, not by understanding.

4. **MMLU is increasingly saturated.** Frontier models score 86-90%. 72% just means the model isn't at the frontier — it doesn't say it's a good OR bad chatbot.

5. **Better eval:** LMSYS Chatbot Arena (human ELO ratings from real conversations), HumanEval for code, TruthfulQA for honesty, plus safety evaluations.

</details>